In [4]:
from pathlib import Path
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    train_path: Path = Path("../data/train.csv")
    test_path: Path = Path("../data/test.csv")
    sub_path: Path = Path("../data/sample_submission.csv")
    pltpd_path: Path = Path("../data/podcast_dataset.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.02

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    max_bin: int = 1024
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG() 
asdict(cfg)

{'train_path': PosixPath('../data/train.csv'),
 'test_path': PosixPath('../data/test.csv'),
 'sub_path': PosixPath('../data/sample_submission.csv'),
 'pltpd_path': PosixPath('../data/podcast_dataset.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.02,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [5]:
from IPython.display import display
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess_df(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    df.loc[df['Episode_Length_minutes']>121.0, 'Episode_Length_minutes'] = 121.0

    df['Host_Guest_Diff'] = df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage']
    df['Host_Guest_Ratio'] = (df['Host_Popularity_percentage'] / df['Guest_Popularity_percentage']).replace([float('inf'), -float('inf')], pd.NA)

    if "Listening_Time_minutes" in df.columns:
        df['Listening_Episode_Diff'] = df['Episode_Length_minutes'] - df['Listening_Time_minutes']
        df['Listening_Episode_Ratio'] = (df['Episode_Length_minutes'] / df['Listening_Time_minutes']).replace([float('inf'), -float('inf')], pd.NA)

    return df


df_train = pd.read_csv(cfg.train_path, index_col='id')
df_test = pd.read_csv(cfg.test_path, index_col='id')
df_sub = pd.read_csv(cfg.sub_path, index_col='id')

df_pltpd = pd.read_csv(cfg.pltpd_path)
df_pltpd = df_pltpd.dropna(subset=['Listening_Time_minutes'])
df_pltpd = df_pltpd.reset_index(drop=True)
df_pltpd.index = df_pltpd.index + 1000000

df_train = pd.concat([df_train, df_pltpd], axis=0)
df_train["id"] = df_train.index

# is_dev_mode = False
# # is_dev_mode = True
# if is_dev_mode:
#     df_train = df_train.sample(10000, random_state=42)
#     df_test = df_test[:10]
#     df_sub = df_sub[:10]
    
df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

# target_col = "Listening_Time_minutes"
# y_train = df_train[target_col].copy()
# df_train = df_train.drop(columns=[target_col])

display(df_train)
display(df_train.describe())
display(df_train.isna().sum())

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
0,0,NaN,0,74.81,3,21,NaN,0.0,2,31.419980,0,98,NaN,NaN,NaN,NaN
1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.012410,1,26,-9.00,0.881501,31.787590,1.361172
2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.925310,2,16,61.00,7.800446,28.974690,1.644952
3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.278240,3,45,-21.48,0.727065,20.891760,1.451438
4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.610310,4,86,21.39,1.364519,34.899690,1.461573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1047100,33,24.81,9,66.15,0,17,98.63,1.0,1,20.573795,1047100,17,-32.48,0.670688,4.236205,1.205903
1047101,11,92.15,6,89.61,5,21,25.82,2.0,0,76.198459,1047101,9,63.79,3.470565,15.951541,1.209342
1047102,23,112.27,1,26.33,5,21,55.29,0.0,1,107.602135,1047102,24,-28.96,0.476216,4.667865,1.043381
1047103,19,NaN,8,41.47,2,14,33.58,0.0,1,17.220998,1047103,85,7.89,1.234961,NaN,NaN


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Listening_Episode_Diff
count,797105.000000,705317.000000,797105.000000,797105.000000,797105.000000,797105.000000,646356.000000,797104.000000,797105.000000,797105.000000,7.971050e+05,797105.000000,646356.000000,705317.000000
mean,23.540988,64.408705,4.554814,59.877839,3.028731,15.663856,52.095246,1.357792,0.998145,45.444668,4.133258e+05,51.378954,7.627507,18.679341
std,13.911304,32.981409,2.962341,22.889880,2.022848,4.027274,28.483819,1.149681,0.815531,27.140915,2.598146e+05,28.131239,36.152160,13.566281
min,0.000000,0.000000,0.000000,1.300000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,-80.170000,-115.540000
25%,12.000000,35.670000,2.000000,39.450000,1.000000,14.000000,28.100000,0.000000,0.000000,23.184220,1.992760e+05,28.000000,-18.280000,8.130000
50%,23.000000,63.770000,5.000000,60.060000,3.000000,17.000000,53.350000,1.000000,1.000000,43.392270,3.985520e+05,52.000000,6.640000,15.643750
75%,36.000000,94.000000,7.000000,79.560000,5.000000,21.000000,76.490000,2.000000,2.000000,64.814620,5.978280e+05,75.000000,33.000000,26.683090
max,47.000000,121.000000,9.000000,119.460000,6.000000,21.000000,119.910000,103.910000,2.000000,119.970000,1.047104e+06,100.000000,113.550000,103.220440


Podcast_Name                        0
Episode_Length_minutes          91788
Genre                               0
Host_Popularity_percentage          0
Publication_Day                     0
Publication_Time                    0
Guest_Popularity_percentage    150749
Number_of_Ads                       1
Episode_Sentiment                   0
Listening_Time_minutes              0
id                                  0
Episode_Num                         0
Host_Guest_Diff                150749
Host_Guest_Ratio               150752
Listening_Episode_Diff          91788
Listening_Episode_Ratio        100172
dtype: int64

In [6]:
cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage']

df_dup = df_train.copy()
df_dup = df_dup.dropna(subset=['Guest_Popularity_percentage'])
df_dup = df_dup[df_dup.duplicated(subset=cols_to_compare, keep=False)]
# df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup = df_dup.sort_values(cols_to_compare)
df_dup

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341
163092,0,93.78,0,68.03,5,10,17.16,1.0,1,71.796010,163092,1,50.87,3.964452,21.983990,1.306201
348103,0,96.02,0,68.03,5,10,17.16,1.0,1,71.796010,348103,1,50.87,3.964452,24.223990,1.3374
216180,0,115.56,0,98.62,4,14,2.48,1.0,0,106.422180,216180,2,96.14,39.766129,9.137820,1.085864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044806,47,66.66,6,38.92,1,14,15.98,1.0,1,47.523106,1044806,98,22.94,2.435544,19.136893,1.402686
123989,47,102.45,6,42.13,6,17,41.29,0.0,1,89.820570,123989,99,0.84,1.020344,12.629430,1.140607
548936,47,102.45,6,42.13,0,14,41.29,0.0,0,89.820570,548936,99,0.84,1.020344,12.629430,1.140607
1035367,47,102.45,6,42.13,6,14,41.29,0.0,1,89.820573,1035367,99,0.84,1.020344,12.629427,1.140607


In [7]:
x = 100
df_dup.iloc[x*50 : (x+1)*50]

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
1045613,10,NaN,7,21.74,4,14,85.22,1.0,1,20.593404,1045613,82,-63.48,0.255104,NaN,NaN
272705,10,NaN,7,22.74,5,17,85.22,1.0,1,20.593400,272705,82,-62.48,0.266839,NaN,NaN
657519,10,NaN,7,22.74,5,14,85.22,1.0,2,20.593400,657519,82,-62.48,0.266839,NaN,NaN
683483,10,NaN,7,22.74,4,14,85.22,0.0,2,20.593400,683483,82,-62.48,0.266839,NaN,NaN
207127,10,49.01,7,24.96,1,21,77.04,1.0,1,27.193530,207127,84,-52.08,0.323988,21.816470,1.802267
1017648,10,49.01,7,24.96,1,21,77.04,3.0,1,27.193532,1017648,84,-52.08,0.323988,21.816468,1.802267
427428,10,98.18,7,33.94,2,14,91.52,2.0,1,66.109610,427428,84,-57.58,0.370848,32.070390,1.485109
1037767,10,96.12,7,33.94,2,14,91.52,2.0,1,66.109612,1037767,84,-57.58,0.370848,30.010388,1.453949
1045022,10,96.12,7,33.94,2,14,91.52,2.0,1,66.109612,1045022,84,-57.58,0.370848,30.010388,1.453949
338180,10,98.20,7,91.04,2,14,91.48,2.0,1,66.109610,338180,84,-0.44,0.99519,32.090390,1.485412


In [8]:
cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage']

df_dup = df_train.copy()
# df_dup = df_dup.dropna(subset=['Guest_Popularity_percentage'])
df_dup = df_dup[df_dup.duplicated(subset=cols_to_compare, keep=False)]
# df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup = df_dup.sort_values(cols_to_compare)
df_dup

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341
340364,0,62.65,0,54.62,1,21,9.30,0.0,2,49.659340,340364,1,45.32,5.873118,12.990660,1.261596
394230,0,62.65,0,54.62,6,21,5.21,0.0,2,49.659340,394230,1,49.41,10.483685,12.990660,1.261596
163092,0,93.78,0,68.03,5,10,17.16,1.0,1,71.796010,163092,1,50.87,3.964452,21.983990,1.306201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
666706,47,37.33,6,29.88,5,21,90.88,0.0,0,36.783300,666706,100,-61.00,0.328785,0.546700,1.014863
407735,47,66.40,6,49.32,3,17,18.25,2.0,0,55.282330,407735,100,31.07,2.702466,11.117670,1.201107
642204,47,67.02,6,49.32,3,17,42.16,2.0,2,55.282330,642204,100,7.16,1.169829,11.737670,1.212322
629922,47,26.28,6,76.28,4,17,28.58,3.0,1,6.548230,629922,100,47.70,2.668999,19.731770,4.013298


In [19]:
x = 110
df_dup.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
556420,1,99,86.62,44.25,4.015990,18.21,1,10,1
601196,1,99,86.62,82.86,31.290830,53.74,5,14,0
321528,1,100,26.19,93.14,3.876570,11.95,2,14,1
340607,1,100,26.19,72.11,3.876570,11.95,2,14,1
68996,1,100,48.49,83.01,7.274330,38.74,3,17,0
1032919,1,100,48.49,83.01,7.274332,10.72,3,17,0
555783,1,100,70.24,85.69,44.831720,94.04,6,21,1
565232,1,100,70.24,24.39,83.607390,104.13,2,10,1
176510,1,100,87.16,32.72,17.570580,NaN,2,10,0
218635,1,100,87.16,89.71,17.570890,32.05,4,10,0


In [11]:
grouped = df_train.groupby(cols_to_compare)
result = grouped.filter(lambda x: x['Listening_Time_minutes'].nunique() > 1)
result = result.sort_values(cols_to_compare)
result

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341
2221,0,55.10,0,68.79,6,14,6.29,1.0,2,35.762540,2221,1,62.50,10.936407,19.337460,1.540718
344896,0,107.40,0,68.79,5,17,17.40,1.0,1,71.796010,344896,1,51.39,3.953448,35.603990,1.495905
410859,0,NaN,0,68.42,6,17,72.02,0.0,2,60.835260,410859,2,-3.60,0.950014,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61038,47,69.20,6,29.32,0,14,84.84,2.0,0,63.532210,61038,100,-55.52,0.345592,5.667790,1.089211
556562,47,37.33,6,29.32,5,21,35.88,0.0,2,36.783300,556562,100,-6.56,0.817168,0.546700,1.014863
1034568,47,65.05,6,29.32,5,10,71.65,2.0,2,36.773310,1034568,100,-42.33,0.409211,28.276690,1.768946
629922,47,26.28,6,76.28,4,17,28.58,3.0,1,6.548230,629922,100,47.70,2.668999,19.731770,4.013298


In [17]:
x = 110
result.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
352993,3,78,23.68,65.11,20.368020,33.99,2,14,0
744644,3,78,23.68,62.24,14.190000,33.67,6,17,1
236720,3,78,23.77,70.85,61.445510,71.30,2,14,1
607678,3,78,23.77,28.41,45.283840,NaN,5,17,1
59971,3,78,41.96,85.37,62.638460,67.51,6,21,2
1025600,3,78,41.96,85.37,62.638462,84.73,6,21,2
198876,3,78,49.44,92.46,8.690000,22.29,3,10,1
552721,3,78,49.44,81.25,8.780000,NaN,3,10,1
677774,3,78,58.95,95.30,78.380890,NaN,1,21,1
1011502,3,78,58.95,95.30,78.380892,NaN,1,21,1


In [20]:
grouped = df_dup.groupby('Listening_Time_minutes')
group_stats = grouped.describe()

group_means = grouped.mean()

group_diffs = group_means.diff()

def calc_all_diffs(group):
    for col in group.select_dtypes(include=['number']).columns:
        if col != 'Listening_Time_minutes':
            group[f'{col}_diff'] = group[col].diff()
    return group

result_with_diffs = df_dup.groupby('Listening_Time_minutes').apply(calc_all_diffs)

# Print results
print("Group Statistics (mean, std, min, max, etc. for all columns):")
display(group_stats)

print("\nDifference between consecutive group means:")
display(group_diffs)

print("\nDataFrame with calculated differences for each value:")
display(result_with_diffs.head())

group_pct_change = group_means.pct_change() * 100

print("\nPercentage change between consecutive group means:")
display(group_pct_change)

Group Statistics (mean, std, min, max, etc. for all columns):


Podcast_Name                                          \
                              count       mean        std   min   25%   50%   
Listening_Time_minutes                                                        
0.000000                       52.0  22.384615  14.501053   2.0   8.0  22.5   
0.001750                        1.0  43.000000        NaN  43.0  43.0  43.0   
0.001755                        1.0  43.000000        NaN  43.0  43.0  43.0   
0.012569                        2.0   4.000000   0.000000   4.0   4.0   4.0   
0.012570                        2.0  22.000000   0.000000  22.0  22.0  22.0   
...                             ...        ...        ...   ...   ...   ...   
117.340000                      2.0  15.000000   0.000000  15.0  15.0  15.0   
117.610000                      3.0  44.000000   0.000000  44.0  44.0  44.0   
117.851650                      2.0  39.000000   0.000000  39.0  39.0  39.0   
119.104670                      2.0  41.000000   0.000000  41.0  41.0  41.0   
119.660000                      3.0  15.000000   0.000000  15.0  15.0  15.0   

                                   Episode_Length_minutes              ...  \
                         75%   max                  count        mean  ...   
Listening_Time_minutes                                                 ...   
0.000000                28.0  47.0                   42.0    7.866905  ...   
0.001750                43.0  43.0                    1.0    5.700000  ...   
0.001755                43.0  43.0                    1.0    5.700000  ...   
0.012569                 4.0   4.0                    2.0    7.000000  ...   
0.012570                22.0  22.0                    2.0    7.795000  ...   
...                      ...   ...                    ...         ...  ...   
117.340000              15.0  15.0                    2.0  118.670000  ...   
117.610000              44.0  44.0                    3.0  118.616667  ...   
117.851650              39.0  39.0                    2.0  119.880000  ...   
119.104670              41.0  41.0                    1.0  119.170000  ...   
119.660000              15.0  15.0                    3.0  118.890000  ...   

                       Host_Guest_Diff        Listening_Episode_Diff  \
                                   75%    max                  count   
Listening_Time_minutes                                                 
0.000000                         35.22  84.18                   42.0   
0.001750                         20.24  20.24                    1.0   
0.001755                         20.24  20.24                    1.0   
0.012569                          6.40   6.40                    2.0   
0.012570                         60.77  60.77                    2.0   
...                                ...    ...                    ...   
117.340000                        7.78   7.78                    2.0   
117.610000                       41.72  41.72                    3.0   
117.851650                      -18.19 -18.19                    2.0   
119.104670                       28.86  28.86                    1.0   
119.660000                       23.75  23.75                    3.0   

                                                                          \
                            mean       std       min       25%       50%   
Listening_Time_minutes                                                     
0.000000                7.866905  1.817129  5.300000  6.340000  7.930000   
0.001750                5.698250       NaN  5.698250  5.698250  5.698250   
0.001755                5.698245       NaN  5.698245  5.698245  5.698245   
0.012569                6.987431  0.000000  6.987431  6.987431  6.987431   
0.012570                7.782430  0.275772  7.587430  7.684930  7.782430   
...                          ...       ...       ...       ...       ...   
117.340000              1.330000  0.000000  1.330000  1.330000  1.330000   
117.610000              1.006667  0.338575  0.620000  


Difference between consecutive group means:


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
Listening_Time_minutes,,,,,,,,,,,,,,,
0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0.001750,20.615385,-2.166905,-3.923077,-28.380769,-2.692308,5.346154,-40.436923,-1.923077,0.153846,-861073.461538,-45.653846,12.056154,0.083767,-2.168655,NaN
0.001755,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,885043.000000,0.000000,0.000000,0.0,-0.000005,-8.976077
0.012569,-39.000000,1.300000,3.000000,38.420000,3.000000,-11.000000,52.260000,0.000000,0.000000,5977.500000,32.000000,-13.840000,-1.260407,1.289186,-2691.241409
0.012570,18.000000,0.795000,0.000000,10.080000,0.000000,0.000000,-44.290000,0.000000,0.000000,-839445.500000,50.000000,54.370000,2.55846,0.794999,63.201916
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117.340000,-29.000000,1.520000,3.000000,-22.210000,3.500000,0.000000,-39.460000,-0.500000,0.000000,-685453.500000,21.000000,17.250000,0.242951,1.330000,0.011335
117.610000,29.000000,-0.053333,-3.000000,14.040000,-5.500000,0.666667,-19.900000,0.500000,-1.333333,-221464.833333,-4.000000,33.940000,1.057896,-0.323333,-0.002775
117.851650,-5.000000,1.263333,0.000000,-54.980000,3.000000,-4.666667,4.930000,0.000000,1.333333,911623.833333,-13.000000,-59.910000,-1.658532,1.021683,0.008652



DataFrame with calculated differences for each value:


Podcast_Name  Episode_Length_minutes  Genre  \
Listening_Time_minutes                                                        
0.0                    1030473             2                     NaN      2   
                       1045286             2                     NaN      2   
                       1004811             2                     NaN      2   
                       1046116             2                     NaN      2   
                       1040583             3                   10.69      3   

                                Host_Popularity_percentage  Publication_Day  \
Listening_Time_minutes                                                        
0.0                    1030473                       67.02                6   
                       1045286                       67.02                6   
                       1004811                       95.12                0   
                       1046116                       95.12                0   
                       1040583                       49.30                0   

                                Publication_Time  Guest_Popularity_percentage  \
Listening_Time_minutes                                                          
0.0                    1030473                21                        74.84   
                       1045286                21                        74.84   
                       1004811                17                        59.90   
                       1046116                17                        59.90   
                       1040583                21                        42.58   

                                Number_of_Ads  Episode_Sentiment  \
Listening_Time_minutes                                             
0.0                    1030473            3.0                  1   
                       1045286            3.0                  1   
                       1004811            2.0                  1   
                       1046116            2.0                  1   
                       1040583            3.0                  1   

                                Listening_Time_minutes  ...  \
Listening_Time_minutes                                  ...   
0.0                    1030473                     0.0  ...   
                       1045286                     0.0  ...   
                       1004811                     0.0  ...   
                       1046116                     0.0  ...   
                       1040583                     0.0  ...   

                                Host_Popularity_percentage_diff  \
Listening_Time_minutes                                            
0.0                    1030473                              NaN   
                       1045286                             0.00   
                       1004811                            28.10   
                       1046116                             0.00   
                       1040583                           -45.82   

                                Publication_Day_diff  Publication_Time_diff  \
Listening_Time_minutes                                                        
0.0                    1030473                   NaN                    NaN   
                       1045286                   0.0                    0.0   
                       1004811                  -6.0                   -4.0   
                       1046116                   0.0                    0.0   
                       1040583                   0.0                    4.0   

                               Guest_Popularity_percentage_diff  \
Listening_Time_minutes                                            
0.0                    1030473                              NaN   
                       1045286                             0.00   
                       1004811                           -14.94   
                       1046116                             0.00   
           


Percentage change between consecutive group means:


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
Listening_Time_minutes,,,,,,,,,,,,,,,
0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0.001750,92.096220,-27.544566,-79.6875,-44.658420,-100.000000,34.152334,-73.034442,-100.000000,18.181818,-85.567121,-91.944229,147.316477,3.687085,-27.566811,NaN
0.001755,0.000000,0.000000,0.0000,0.000000,NaN,0.000000,0.000000,inf,0.000000,609.365877,0.000000,0.000000,0.000000,-0.000085,-0.275581
0.012569,-90.697674,22.807018,300.0000,109.240830,inf,-52.380952,350.033490,0.000000,0.000000,0.580180,800.000000,-68.379447,-53.505498,22.624260,-82.854163
0.012570,450.000000,11.357143,0.0000,13.697513,0.000000,0.000000,-65.917547,0.000000,0.000000,-81.007189,138.888889,849.531250,233.595455,11.377558,11.348364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117.340000,-65.909091,1.297482,inf,-26.237448,175.000000,0.000000,-41.925202,-50.000000,0.000000,-66.280287,56.756757,-182.154171,27.013014,inf,1.133458
117.610000,193.333333,-0.044943,-100.0000,22.485586,-100.000000,4.761905,-36.406879,100.000000,-66.666667,-63.507833,-6.896552,436.246787,92.608232,-24.310777,-0.274412
117.851650,-11.363636,1.065056,NaN,-71.888075,inf,-31.818182,14.182969,0.000000,200.000000,716.371897,-24.074074,-143.600192,-75.379932,101.491722,0.857826


In [ ]:
cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage']

df_test_with_id = df_test.copy()
df_test_with_id = df_test_with_id.dropna(subset=['Guest_Popularity_percentage'])

leaked_rows = df_test_with_id.merge(
    df_train[cols_to_compare + ['Listening_Time_minutes']].drop_duplicates(),
    on=cols_to_compare,
    how='inner'
)
leaked_rows

# mean_values = leaked_rows.groupby('id')['Listening_Time_minutes'].mean().reset_index()

# display(df_test.loc[df_test['id'].isin(mean_values['id'].values)])
# df_test.loc[df_test['id'].isin(mean_values['id'].values), "Listening_Time_minutes2"] = mean_values["Listening_Time_minutes"].values
# display(df_test.loc[df_test['id'].isin(mean_values['id'].values)])

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,id,Listening_Time_minutes
0,27,66.28,4,50.26,6,14,65.14,2.0,0,58,-14.88,0.771569,750052,65.789560
1,27,66.28,4,50.26,6,14,65.14,2.0,0,58,-14.88,0.771569,750052,65.789564
2,31,105.62,8,72.92,4,21,70.95,0.0,2,28,1.97,1.027766,750384,98.168340
3,36,114.01,2,80.18,0,14,22.72,0.0,1,2,57.46,3.529049,750542,89.175290
4,36,114.01,2,80.18,0,14,22.72,0.0,1,2,57.46,3.529049,750542,89.175292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6400,3,53.68,3,83.53,3,21,20.06,2.0,1,9,63.47,4.164008,999780,48.214100
6401,3,100.33,3,97.53,0,14,2.76,1.0,2,85,94.77,35.336957,999792,61.030350
6402,47,108.74,6,37.72,6,14,26.91,0.0,2,74,10.81,1.401709,999832,89.819210
6403,46,34.12,9,80.03,4,21,97.79,2.0,2,64,-17.76,0.818386,999841,17.234690


In [43]:
leaked_rows

,id,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes_x,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_y
0,1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.012410,26,-9.00,0.881501,31.787590,1.361172,88.01241
1,2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.925310,16,61.00,7.800446,28.974690,1.644952,44.92531
2,3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.278240,45,-21.48,0.727065,20.891760,1.451438,46.27824
3,4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.610310,86,21.39,1.364519,34.899690,1.461573,75.61031
4,6,6,69.83,0,35.82,6,21,39.02,0.0,1,64.750240,47,-3.20,0.917991,5.079760,1.078452,64.75024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608777,1047064,32,104.55,4,25.84,3,10,88.89,0.0,1,84.539392,37,-63.05,0.290696,20.010608,1.236702,84.53939
608778,1047066,44,19.15,0,69.81,5,10,0.70,3.0,2,18.695367,51,69.11,99.728571,0.454633,1.024318,18.69537
608779,1047070,19,NaN,8,74.49,3,21,64.94,1.0,2,24.419968,86,9.55,1.147059,NaN,NaN,24.41997
608780,1047073,5,54.63,4,35.80,2,21,88.80,0.0,2,53.978371,25,-53.00,0.403153,0.651629,1.012072,53.97837
